In [ ]:
# Import necessary libraries
import requests
from tqdm import tqdm
import re
import pandas as pd 
from bert_score import score
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import ast

# Evaluation

==========================================
🧮 Evaluation Methods
------------------------------------------
1️⃣ Lexical Comparison:
    • BERTScore (P/R/F1) using DeBERTa-xlarge-mnli
    • Semantic Similarity via SentenceTransformer embeddings
2️⃣ LLM-as-a-Judge:
    • Qualitative assessment by a LLM
      Evaluating models responses based on completeness, correctness, contextual relevance, and clarity.
==========================================

In [22]:
df = pd.read_csv('/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/Paper/Data/Sample Data_responses.csv')

In [23]:
'''
==========================================
Evaluation: BERTScore + Semantic Similarity
------------------------------------------
Compares model responses against instructor answers: (You need to change Model 1 to other Models as well)
  • BERTScore (Precision/Recall/F1) via microsoft/deberta-xlarge-mnli
  • SentenceTransformer cosine similarity via all-MiniLM-L6-v2

Writes metrics to:
  df['Model 1_bertscore_precision'], df['Model 1_bertscore_recall'],
  df['Model 1_bertscore_f1'], df['Model 1_semantic_similarity']

Skips rows with empty candidate/reference.
==========================================

'''

bert_model = "microsoft/deberta-xlarge-mnli"  # Better BERTScore performance
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize new columns
df['bertscore_precision'] = None
df['bertscore_recall'] = None
df['bertscore_f1'] = None
df['semantic_similarity'] = None

# Compute metrics row by row
for i in tqdm(df.index):
    candidate = str(df.at[i, 'Model 1'])
    reference = str(df.at[i, 'instructor_answer'])

    # Skip empty values
    if not candidate.strip() or not reference.strip():
        continue

    # --- BERTScore ---
    P, R, F1 = score([candidate], [reference], lang='en', model_type=bert_model, verbose=False)
    df.at[i, 'Model 1_bertscore_precision'] = round(P.item(), 4)
    df.at[i, 'Model 1_bertscore_recall'] = round(R.item(), 4)
    df.at[i, 'Model 1_bertscore_f1'] = round(F1.item(), 4)

    # --- SentenceTransformer Similarity ---
    emb1 = st_model.encode(candidate, convert_to_tensor=True)
    emb2 = st_model.encode(reference, convert_to_tensor=True)
    sim_score = util.pytorch_cos_sim(emb1, emb2).item()
    df.at[i, 'Model 1_semantic_similarity'] = round(sim_score, 4)


100%|██████████| 10/10 [00:32<00:00,  3.25s/it]


# LLM as a JUDGE

In [ ]:
# API config
api_key = "#"
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}
endpoint = "https://#"
deployment_id = "#-gpt-4o"
api_version = "2025-01-01-preview"


judgment_cols = [
    "judge_model1_correctness","judge_model1_completeness","judge_model1_relevance","judge_model1_clarity","judge_model1_final_score",
    "judge_model2_correctness","judge_model2_completeness","judge_model2_relevance","judge_model2_clarity","judge_model2_final_score",
    "judge_model3_correctness","judge_model3_completeness","judge_model3_relevance","judge_model3_clarity","judge_model3_final_score",
    "judge_model4_correctness","judge_model4_completeness","judge_model4_relevance","judge_model4_clarity","judge_model4_final_score",
    "judge_comparison_summary"
]
for col in judgment_cols:
    if col not in df.columns:
        df[col] = None

# ==== Helpers for actions ====
def parse_actions(raw):
    """
    raw: list like [[verb, noun, step], ...] or string repr.
    returns flat tokens: ["rotate lens", "press release_button", ...]
    """
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    if isinstance(raw, str):
        try:
            raw = ast.literal_eval(raw)
        except Exception:
            return []
    out = []
    for item in (raw or []):
        if isinstance(item, (list, tuple)) and len(item) == 3:
            verb, noun, _ = item
            if verb and noun:
                out.append(f"{verb} {noun}")
    return out

def dedupe_consecutive(seq):
    """Collapse immediate repeats from frame jitter."""
    out, prev = [], None
    for x in seq:
        if x != prev:
            out.append(x)
            prev = x
    return out

def format_actions(seq, omitted=0):
    """One-liner with optional omitted count."""
    if not seq:
        return "No hand actions detected."
    txt = ", ".join(seq) + "."
    if omitted > 0:
        txt += f"  (+{omitted} more)"
    return txt

# ==== Conversation history builder ====
def build_conversation_history(qa_history, current_time):
    text = "\n".join(
        f'Student: "{q}" at {t} sec\nInstructor: "{a}"'
        for t, q, a in qa_history if t < current_time
    )
    return text if text else "No prior conversation. This is the first question."

# ==== Judge prompt builder (UNGROUPED; earlier capped, recent uncapped) ====
def build_judge_prompt_multi(
    task_type,
    task_duration,
    task_description,
    question_time,
    conversation_history,
    task_step,
    hand_actions,         # earlier: start → last question
    hand_actions_now,     # recent: last question → now
    question,
    instructor_answer,    # may be empty
    ai_answers,           # dict {"Model 1": "...", ..., "Model 4": "..."}
    include_reference=True,
    randomize_models=False,
    cap_earlier=80,
    dedupe_recent=True,
):
    # Parse & prep actions
    earlier = dedupe_consecutive(parse_actions(hand_actions))
    recent  = parse_actions(hand_actions_now)
    if dedupe_recent:
        recent = dedupe_consecutive(recent)

    # Cap earlier to most recent N; do NOT cap recent
    if cap_earlier is not None and len(earlier) > cap_earlier:
        earlier_kept = earlier[-cap_earlier:]
        earlier_omitted = len(earlier) - cap_earlier
    else:
        earlier_kept = earlier
        earlier_omitted = 0

    earlier_text = format_actions(earlier_kept, earlier_omitted)
    recent_text  = format_actions(recent, omitted=0)

    # Optionally randomize model order (names preserved)
    items = list(ai_answers.items())
    # if randomize_models:
    #     random.shuffle(items)

    # Build prompt
    parts = []
    parts.append("You are an expert judge evaluating and comparing four AI-generated responses to a student’s question during a technical task.\n")
    parts.append("---\n")
    parts.append(f"Task Type: {task_type}")
    parts.append(f"Total Task Duration: {int(task_duration)} seconds")
    parts.append(f"Task Steps (may vary by session): {task_description}")
    parts.append(f"Time of Student Question: {int(question_time)} seconds\n")

    parts.append("Conversation History:")
    parts.append(conversation_history if conversation_history else "(None)")
    parts.append("")

    if task_step and task_step != "No coarse-grained action":
        parts.append(f"Current detected task step: {task_step}\n")
    elif task_step == "No coarse-grained action":
        parts.append("The current step of the task is not specified. The student may not have started yet, or the detection model failed to identify it.\n")

    parts.append("Hand actions observed earlier in the task (start → last question):")
    parts.append(earlier_text + "\n")

    parts.append(f"Most recent hand actions (last question → now @ {int(question_time)}s):")
    parts.append(recent_text + "\n")

    parts.append("---\n")
    parts.append(f'Current Student Question (asked at {int(question_time)} seconds):\n"{question}"\n')

    if include_reference and isinstance(instructor_answer, str) and instructor_answer.strip():
        parts.append("Reference instructor reply (may be imperfect; do NOT require a match):")
        parts.append(instructor_answer.strip())
        parts.append("")

    parts.append("AI Responses:")
    for model_name, response in items:
        parts.append(f"\n- {model_name}:\n{(response or '').strip()}")

    parts.append("""
---
### Evaluation Instructions

Judge the responses using ONLY the shared context above. The candidate order may be randomized; ignore position and verbosity.

Score each model on:
- Correctness: 0–5 (5 = fully accurate; 3 = minor issues; 1 = incorrect aspects; 0 = wrong)
- Completeness: 0–5 (5 = fully answers; 3 = partial; 1 = minimal; 0 = none)
- Contextual Relevance: 0–5 (5 = fully grounded in the provided context/hand actions; 3 = weakly grounded; 1 = irrelevant; 0 = off-topic)
- Clarity: 0–5 (5 = very clear; 3 = somewhat clear; 1 = vague; 0 = incomprehensible)
- Final Score: 0–5 (holistic judgment; NOT an average)

Return ONLY this format for each model:
Model: Model N
Correctness: X/5
Completeness: X/5
Contextual Relevance: X/5
Clarity: X/5
Final Score: X/5

Do not include per-model explanations.
At the end, add a brief Comparison Summary (1–3 sentences) explaining which model(s) performed best and why.

Winner: Model N
Comparison Summary: ...
""")

    return "\n".join(parts)

# ==== Evaluation loop ====
contents = []
for idx in tqdm(df.index):
    try:
        row = df.loc[idx]
        if pd.isna(row.get("question_time", None)):
            continue

        current_time = row["question_time"]
        video = row["video_name"]

        # Earlier rows (same video, earlier questions)
        earlier_rows = df[(df["video_name"] == video) & (df["question_time"] < current_time)]

        # Conversation history uses instructor_answer from earlier rows
        qa_history = list(zip(
            earlier_rows["question_time"],
            earlier_rows["question"],
            earlier_rows["instructor_answer"]
        ))
        conversation_history = build_conversation_history(qa_history, current_time)

        # AI answers for the four models, you can consider different permutations here
        ai_answers = {
            "Model 1": row.get("Model 1", ""),
            "Model 2": row.get("Model 2", ""),
            "Model 3": row.get("Model 3", ""),
            "Model 4": row.get("Model 4", "")
        }

        # Aggregate earlier hand actions (all prior rows in the same video)
        all_prev_hand_actions = []
        for _, r in earlier_rows.iterrows():
            if 'hand_actions_data' in r and not pd.isna(r['hand_actions_data']):
                try:
                    ha = ast.literal_eval(r['hand_actions_data']) if isinstance(r['hand_actions_data'], str) else r['hand_actions_data']
                    all_prev_hand_actions.extend(ha or [])
                except Exception as e:
                    print(f"❌ Error parsing hand actions at row {r.name}: {e}")

        # Current row's hand actions (recent block)
        hand_actions_now = []
        if 'hand_actions_data' in row and not pd.isna(row['hand_actions_data']):
            try:
                hand_actions_now = ast.literal_eval(row['hand_actions_data']) if isinstance(row['hand_actions_data'], str) else row['hand_actions_data']
            except Exception as e:
                print(f"❌ Error parsing current hand actions at row {idx}: {e}")
                hand_actions_now = []

        # Build judge prompt
        judge_prompt = build_judge_prompt_multi(
            task_type=row.get("task_type", ""),
            task_duration=row.get("task_duration", 0),
            task_description=row.get("task_description", ""),
            question_time=current_time,
            conversation_history=conversation_history,
            task_step=row.get("task_step", ""),
            hand_actions=all_prev_hand_actions,   # earlier (start → last Q)
            hand_actions_now=hand_actions_now,    # recent (last Q → now)
            question=row.get("question", ""),
            instructor_answer=row.get("instructor_answer", ""),
            ai_answers=ai_answers,
            include_reference=True,
            randomize_models=False,               # set True if you want to mitigate position bias (other ways: permutations)
            cap_earlier=80,                       # keep most recent 80 earlier actions
            dedupe_recent=True                    # de-jitter recent; set False if you want raw
        )

        # print(judge_prompt)
        # Call LLM as judge
        data = {
            "messages": [{"role": "user", "content": judge_prompt}],
            "temperature": 0.5,
            "max_tokens": 1000
        }

        response = requests.post(
            f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
            headers=headers,
            json=data
        )

        result = response.json()
        content = result.get("choices", [{}])[0].get("message", {}).get("content", "")
        contents.append(content)
        # print(result)

        # ---- Parse scores for each model ----
        for i in range(1, 5):
            # Accept either "Model: Model i" or "**Model i**" variants
            model_tag = rf"(?:\*\*Model {i}\*\*|Model:\s*Model {i})"
            pattern = (
                rf"{model_tag}.*?"
                r"Correctness:\s*(\d(?:\.\d+)?)\s*/\s*5\s*.*?"
                r"Completeness:\s*(\d(?:\.\d+)?)\s*/\s*5\s*.*?"
                r"Contextual Relevance:\s*(\d(?:\.\d+)?)\s*/\s*5\s*.*?"
                r"Clarity:\s*(\d(?:\.\d+)?)\s*/\s*5\s*.*?"
                r"Final Score:\s*(\d+(?:\.\d+)?)\s*/\s*5"
            )
            match = re.search(pattern, content, re.DOTALL | re.IGNORECASE)
            if match:
                df.at[idx, f"judge_model{i}_correctness"]   = float(match.group(1))
                df.at[idx, f"judge_model{i}_completeness"]  = float(match.group(2))
                df.at[idx, f"judge_model{i}_relevance"]     = float(match.group(3))
                df.at[idx, f"judge_model{i}_clarity"]       = float(match.group(4))
                df.at[idx, f"judge_model{i}_final_score"]   = float(match.group(5))
            else:
                print(f"⚠️ Parse failed at row {idx} for Model {i}.")
                # Optionally store raw content for debugging:
                # dfff.at[idx, f"judge_model{i}_final_score"] = None

        winner_match = re.search(r"(?:Winner|Best\s*model)\s*:\s*(?:Model\s*)?(\d)\b",
                         content, re.IGNORECASE)
        if winner_match:
            winner_num = int(winner_match.group(1))
            df.at[idx, "judge_winner"] = f"Model {winner_num}"
        else:    
            df.at[idx, "judge_winner"] = "No Winner"

        # ---- Extract comparison summary ----
        summary_match = re.search(r"Comparison Summary:\s*(.*)", content, re.DOTALL | re.IGNORECASE)
        if summary_match:
            df.at[idx, "judge_comparison_summary"] = summary_match.group(1).strip()

    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        df.at[idx, "judge_comparison_summary"] = "[API or parsing error]"

# Optional: save
# dfff.to_csv("judged_responses_multi_model.csv", index=False)
# print("✅ Evaluation complete and saved.")

